# Day 8 Battery Data Quality Assessment

**Tasks for today**
1. Audit cells/cycles, missing values, invalid readings and target consistency.
2. Check sequence/cycle lengths and partial cycles.
3. Identify future information leakage risks.
4. Propose cleaning/alignment rules.

**Outputs:** Battery data-quality table · Cleaning/alignment plan


## Setup Mount Drive & Locate Dataset

One reusable loader instead of repeating the same drive-mount / unzip / metadata-search logic in every cell.

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd

RAW_ZIP_CANDIDATES = ["/content/drive/MyDrive/**/cleaned_dataset*.zip"]
EARLY_WINDOW_SEC = 500

def locate_dataset(force_remount=False):
    """Mounts Drive (if needed), unzips the dataset (if needed), and returns
    (metadata_path, data_csv_dir, df_meta) with a sanitized 'clean_capacity' column.
    Safe to call multiple times — does nothing if the dataset is already extracted.
    """
    meta_files = [
        os.path.join(r, f)
        for r, _, fs in os.walk("/content")
        if "drive" not in r
        for f in fs if f == "metadata.csv"
    ]

    if not meta_files or force_remount:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=force_remount)

        zip_path = None
        if os.path.exists("/content/cleaned_dataset.zip"):
            zip_path = "/content/cleaned_dataset.zip"
        else:
            for pattern in RAW_ZIP_CANDIDATES:
                hits = glob.glob(pattern, recursive=True)
                if hits:
                    zip_path = hits[0]
                    break
        if zip_path is None:
            raise FileNotFoundError("cleaned_dataset.zip not found in /content or Drive.")

        print(f"Using zip: {zip_path}")
        os.system(f'unzip -o -q "{zip_path}" -d /content/')

        meta_files = [
            os.path.join(r, f)
            for r, _, fs in os.walk("/content")
            if "drive" not in r
            for f in fs if f == "metadata.csv"
        ]
        if not meta_files:
            raise FileNotFoundError("metadata.csv still not found after extraction.")

    metadata_path = meta_files[0]
    data_csv_dir = os.path.join(os.path.dirname(metadata_path), "data")

    df_meta = pd.read_csv(metadata_path)
    df_meta["clean_capacity"] = pd.to_numeric(
        df_meta["Capacity"].astype(str).str.replace(r"[\[\]]", "", regex=True),
        errors="coerce",
    )

    df_meta.loc[df_meta["clean_capacity"] <= 0.05, "clean_capacity"] = np.nan

    return metadata_path, data_csv_dir, df_meta


METADATA_PATH, DATA_CSV_DIR, df_meta = locate_dataset()
print(f"Metadata: {METADATA_PATH}")
print(f"Data dir: {DATA_CSV_DIR}")
print(f"Rows in metadata: {len(df_meta)}")


Mounted at /content/drive
Using zip: /content/drive/MyDrive/cleaned_dataset.zip


## Task 1 Audit Cells, Cycles, Missing Values & Target Consistency

Per battery cell: how many cycles of each type exist, how many discharge cycles have a
usable (non-missing, non-degenerate) capacity target, and whether any capacity readings
look physically implausible (spikes above rated capacity, or unrealistically deep readings).

In [ ]:
def generate_quality_table(df):
    records = []
    for battery_id, grp in df.groupby("battery_id"):
        dis_grp = grp[grp["type"] == "discharge"]
        chg_grp = grp[grp["type"] == "charge"]
        imp_grp = grp[grp["type"] == "impedance"]

        valid_caps = dis_grp["clean_capacity"].dropna()
        missing_count = len(dis_grp) - len(valid_caps)
        temps = "/".join(map(str, sorted(grp["ambient_temperature"].unique())))

        min_c = f"{valid_caps.min():.3f}" if len(valid_caps) else "N/A"
        max_c = f"{valid_caps.max():.3f}" if len(valid_caps) else "N/A"

        issues = []
        if missing_count > 0:
            issues.append(f"{missing_count} missing/aborted")
        if len(valid_caps) and valid_caps.max() > 2.2:
            issues.append("Capacity > 2.2Ah spike (likely sensor fault)")
        if len(valid_caps) and valid_caps.min() < 0.2:
            issues.append("Deep EOL reading (<0.2Ah)")

        records.append({
            "Battery_ID": battery_id,
            "Ambient_Temp_C": temps,
            "Total_Cycles": len(grp),
            "Discharge_Count": len(dis_grp),
            "Charge_Count": len(chg_grp),
            "Impedance_Count": len(imp_grp),
            "Valid_Capacity_Cycles": len(valid_caps),
            "Missing_Cap_Count": missing_count,
            "Min_Capacity_Ah": min_c,
            "Max_Capacity_Ah": max_c,
            "Data_Quality_Notes": ", ".join(issues) if issues else "Clean",
        })
    return pd.DataFrame(records)


df_quality = generate_quality_table(df_meta)
df_quality.to_csv("/content/battery_data_quality_report.csv", index=False)
display(df_quality)


## Task 2 Sequence / Cycle Length & Partial-Cycle Check

The frozen pipeline only uses the **first 500 seconds** of each discharge cycle (see Day 22
methodology). A cycle shorter than that cannot supply a full early window feature set. This
section measures every discharge file's actual duration and row count, and flags short/partial
cycles before they reach feature extraction.

In [ ]:
def audit_cycle_lengths(df, data_dir, early_window_sec=EARLY_WINDOW_SEC):
    dis_meta = df[df["type"] == "discharge"].copy()
    records = []

    for _, r in dis_meta.iterrows():
        fpath = os.path.join(data_dir, r["filename"])
        if not os.path.exists(fpath):
            records.append({"battery_id": r["battery_id"], "test_id": r["test_id"],
                             "status": "file_missing", "duration_sec": np.nan, "n_rows": 0})
            continue
        try:
            raw = pd.read_csv(fpath, usecols=["Time"])
        except Exception:
            records.append({"battery_id": r["battery_id"], "test_id": r["test_id"],
                             "status": "unreadable", "duration_sec": np.nan, "n_rows": 0})
            continue

        duration = raw["Time"].iloc[-1] if len(raw) else 0
        n_rows = len(raw)
        if n_rows < 10:
            status = "too_few_rows"
        elif duration < early_window_sec:
            status = "partial_cycle"  # shorter than the required early window
        else:
            status = "ok"

        records.append({"battery_id": r["battery_id"], "test_id": r["test_id"],
                         "status": status, "duration_sec": duration, "n_rows": n_rows})

    return pd.DataFrame(records)


df_cycle_lengths = audit_cycle_lengths(df_meta, DATA_CSV_DIR)
df_cycle_lengths.to_csv("/content/cycle_length_audit.csv", index=False)

status_summary = df_cycle_lengths["status"].value_counts()
print("Cycle length audit summary:")
print(status_summary)
print(f"\nTotal discharge cycles: {len(df_cycle_lengths)}")
print(f"Usable (status == 'ok'): {(df_cycle_lengths['status'] == 'ok').sum()}")

display(df_cycle_lengths[df_cycle_lengths["status"] != "ok"].head(20))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4.5))
plt.hist(df_cycle_lengths["duration_sec"].dropna(), bins=40, color="#3366cc", edgecolor="black", linewidth=0.4)
plt.axvline(EARLY_WINDOW_SEC, color="red", linestyle="--", linewidth=1.5, label=f"{EARLY_WINDOW_SEC}s cutoff")
plt.xlabel("Discharge Cycle Duration (seconds)")
plt.ylabel("Number of Cycles")
plt.title("Distribution of Discharge Cycle Durations")
plt.legend()
plt.tight_layout()
plt.show()


**Reading this:** any cycle left of the red line cannot supply a full 500-second early window
and must be excluded before feature extraction, not silently zero padded padding would
fabricate signal that never existed and bias the learned features.

## Task 3 Future-Information Leakage Risk Audit

In [ ]:
def extract_early_window_features(file_path, early_window_sec=EARLY_WINDOW_SEC):
    """Builds features ONLY from rows with Time <= early_window_sec.
    This is the single function responsible for enforcing the no-future-information rule.
    """
    if not os.path.exists(file_path):
        return None
    try:
        raw_df = pd.read_csv(file_path)
    except Exception:
        return None

    if len(raw_df) < 10 or "Time" not in raw_df.columns or raw_df["Time"].iloc[-1] < 60:
        return None

    early_df = raw_df[raw_df["Time"] <= early_window_sec].copy()
    if len(early_df) < 5:
        return None

    v = early_df["Voltage_measured"].values
    i = early_df["Current_measured"].values
    t = early_df["Temperature_measured"].values
    time_pts = early_df["Time"].values

    dt_span = time_pts[-1] - time_pts[0] + 1e-6
    v_drop = v[0] - v[min(3, len(v) - 1)]
    r_est = v_drop / (abs(i[min(3, len(i) - 1)]) + 1e-6)

    return {
        "feat_R_est": r_est,
        "feat_V_drop_init": v_drop,
        "feat_dV_dt_early": (v[-1] - v[0]) / dt_span,
        "feat_dT_dt_early": (t[-1] - t[0]) / dt_span,
        "feat_V_mean_early": float(np.mean(v)),
        "feat_V_std_early": float(np.std(v)),
        "feat_T_mean_early": float(np.mean(t)),
        "feat_T_rise_early": float(t[-1] - t[0]),
    }

sample_file = None
for _, r in df_meta[df_meta["type"] == "discharge"].iterrows():
    candidate = os.path.join(DATA_CSV_DIR, r["filename"])
    if os.path.exists(candidate):
        sample_file = candidate
        break

if sample_file:
    raw_sample = pd.read_csv(sample_file)
    max_time_seen_by_extractor = raw_sample[raw_sample["Time"] <= EARLY_WINDOW_SEC]["Time"].max()
    assert max_time_seen_by_extractor <= EARLY_WINDOW_SEC, "Leakage: extractor read past the cutoff!"
    print(f"[PASS] Feature extractor never reads past t={EARLY_WINDOW_SEC}s "
          f"(max time actually used: {max_time_seen_by_extractor:.1f}s)")
else:
    print("[SKIP] No sample discharge file found to verify against.")


In [ ]:
feature_cols_example = [k for k in (extract_early_window_features(sample_file) or {}).keys()]
leaky_cols = [c for c in feature_cols_example if "capacity" in c.lower()]
if leaky_cols:
    print(f"[FAIL] Target-like columns found in features: {leaky_cols}")
else:
    print("[PASS] No capacity/target-derived column present in the feature set:")
    print(f"       {feature_cols_example}")


In [ ]:
from sklearn.model_selection import GroupKFold

def verify_group_split_is_leak_free(groups, n_splits=4):
    gkf = GroupKFold(n_splits=n_splits)
    dummy_X = np.zeros((len(groups), 1))
    seen_in_val = set()
    overlap_found = False

    for fold, (train_idx, val_idx) in enumerate(gkf.split(dummy_X, groups=groups)):
        train_cells = set(groups[train_idx])
        val_cells = set(groups[val_idx])
        overlap = train_cells & val_cells
        if overlap:
            overlap_found = True
            print(f"[FAIL] Fold {fold+1}: overlap between train and val cells -> {overlap}")
        seen_in_val |= val_cells

    if not overlap_found:
        print(f"[PASS] No battery_id appears in both train and validation within any fold "
              f"({n_splits} folds checked).")
    return not overlap_found


_ = verify_group_split_is_leak_free(df_meta["battery_id"].values, n_splits=4)


**Risks identified and how each is controlled:**

| Risk | Where it could enter | Control applied |
|---|---|---|
| Reading beyond the decision point | Using the full discharge curve instead of the early window | `extract_early_window_features` hard-filters on `Time <= 500s`; verified above |
| Target leaking into inputs | Accidentally including capacity-derived columns as features | Feature dict checked programmatically for capacity-related keys |
| Cell identity leakage | Same battery's cycles split across train and validation | `GroupKFold` on `battery_id`; verified with an explicit overlap check |
| Global statistics leakage (flagged for later days) | Fitting a scaler on the full dataset before splitting | Scaler must be fit **inside** the training fold only — rule carried into Day 11/12 preprocessing, not yet implemented in this notebook |


## Task 4 Proposed Cleaning / Alignment Rules

## Applying the Rules Feature Extraction (Preview)

In [ ]:
valid_discharge_meta = df_meta[(df_meta["type"] == "discharge") & (df_meta["clean_capacity"].notna())]
feature_rows = []

for _, r in valid_discharge_meta.iterrows():
    fpath = os.path.join(DATA_CSV_DIR, r["filename"])
    feats = extract_early_window_features(fpath, early_window_sec=EARLY_WINDOW_SEC)
    if feats is None:
        continue  # silently-dropped partial/invalid cycles are already accounted for in Task 2's audit
    feats.update({
        "battery_id": r["battery_id"],
        "test_id": int(r["test_id"]),
        "ambient_temperature": r["ambient_temperature"],
        "target_capacity": float(r["clean_capacity"]),
    })
    feature_rows.append(feats)

df_ml = pd.DataFrame(feature_rows)
df_ml.to_csv("/content/cleaned_ml_ready_battery_dataset.csv", index=False)
print("SHAPE:", df_ml.shape)
display(df_ml.head())


## Leak-Safe Split & Visualization

In [ ]:
gkf = GroupKFold(n_splits=4)
groups = df_ml["battery_id"].values
X = df_ml[[c for c in df_ml.columns if c.startswith("feat_")]]
y = df_ml["target_capacity"].values

print("4-fold group-based split (no cell-identity leakage):")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    val_cells = sorted(set(groups[val_idx]))
    print(f"  Fold {fold+1} — validation cells: {val_cells} ({len(val_idx)} samples)")

plt.figure(figsize=(10, 5))
for cell in sorted(df_ml["battery_id"].unique()):
    sub = df_ml[df_ml["battery_id"] == cell]
    plt.plot(sub["test_id"], sub["target_capacity"], marker=".", label=f"Cell {cell}")

plt.xlabel("Cycle / Test ID")
plt.ylabel("Discharge Capacity (Ah)")
plt.title("Cleaned & Leak-Free Capacity Fade Curves")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
